# Compare IPAB prices

IPAB prices are telling that on haverage the cost of houses went down from 2008.

Compared to other indices, IPAB prices computed by ISTAT, considering:
- stratifications
- edonic adjustments - if people get used to require higher standards for buildings, the price of lower standard buildings will go down by definition

In [2]:
import warnings 
from istatapi import discovery, retrieval
import requests
import pandas as pd

warnings.filterwarnings('ignore')
requests.urllib3.disable_warnings()

In [15]:
ds_structure_id = "DCSP_IPAB"
ds = discovery.DataSet(dataflow_identifier=ds_structure_id)
ds_info = ds.dimensions_info()
print(ds_info.to_markdown(index=False))
info_dicts = {}
for col in ds_info.dimension:
    print(f" >> {col}")
    print(ds.get_dimension_values(col).sort_values("values_ids").to_markdown(index=False))
    info_dicts[col] = ds.get_dimension_values(col).set_index("values_ids")["values_description"].to_dict()

| dimension    | dimension_ID   | description            |
|:-------------|:---------------|:-----------------------|
| FREQ         | CL_FREQ        | Frequency              |
| ITTER107     | CL_ITTER107    | Territory              |
| MISURA1      | CL_MISURA1     | MISURA1                |
| IND_TYPE     | CL_TIPO_DATO2  | Data type 2            |
| ABIT_COMPRAV | CL_TYPPURCH    | Purchases of dwellings |
 >> FREQ
| values_ids   | values_description   |
|:-------------|:---------------------|
| A            | annual               |
| Q            | quarterly            |
 >> ITTER107
| values_ids   | values_description   |
|:-------------|:---------------------|
| IT           | Italy                |
| ITC          | Nord-ovest           |
| ITC1         | Piemonte             |
| ITC11        | Torino               |
| ITC4         | Lombardia            |
| ITC45        | Milano               |
| ITD          | Nord-est             |
| ITE          | Centro (I)           |
| ITE

In [16]:
info_dicts

{'FREQ': {'A': 'annual', 'Q': 'quarterly'},
 'ITTER107': {'IT': 'Italy',
  'ITFG': 'Mezzogiorno',
  'ITE': 'Centro (I)',
  'ITE43': 'Roma',
  'ITE4': 'Lazio',
  'ITC': 'Nord-ovest',
  'ITC11': 'Torino',
  'ITC1': 'Piemonte',
  'ITC45': 'Milano',
  'ITC4': 'Lombardia',
  'ITD': 'Nord-est'},
 'MISURA1': {'4': 'index number',
  '6': 'percentage changes on the previous period',
  '7': 'percentage changes on the same period of the previous year',
  '8': 'annual average rate of change',
  '22': 'not applicable'},
 'IND_TYPE': {'18': 'house price index (base 2010=100) - quarterly data',
  '19': 'house price index (base 2010=100) - annual average',
  '20': 'house price index (base 2010=100) - weights',
  '59': 'house price index (base 2015=100) - quarterly data',
  '60': 'house price index (base 2015=100) - annual average'},
 'ABIT_COMPRAV': {'ALL': 'H1 - all items',
  'EXST_DW': 'H12 - existing dwellings',
  'NEW_DW': 'H11 - new dwellings'}}

In [3]:
ds = discovery.DataSet(dataflow_identifier="DCSP_IPAB") 
#ds.set_filters(freq="Q", itter107=["IT", "ITC", "ITC45"], misura1="4", ind_type="59", abit_comprav=["EXST_DW", "NEW_DW"]) # ITC45 - Milano
df1 = retrieval.get_data(ds) 

In [20]:
df2 = df1.copy()
for col in info_dicts:
    df2[col] = df2[col].astype(str).map(info_dicts[col])

In [37]:
df3 = (
    df2
    .sort_values(by="TIME_PERIOD")
    .query("FREQ == 'quarterly'")
    .query("MISURA1 == 'index number'")
    #.query("ABIT_COMPRAV == 'H12 - existing dwellings'")
    .query("IND_TYPE == 'house price index (base 2015=100) - quarterly data'")
    .dropna(axis=1, how='all')
)
df3

,DATAFLOW,FREQ,ITTER107,MISURA1,IND_TYPE,ABIT_COMPRAV,TIME_PERIOD,OBS_VALUE,OBS_STATUS
4167,IT1:143_497(1.2),quarterly,Roma,index number,house price index (base 2015=100) - quarterly ...,H1 - all items,2010-01-01,129.7,NaN
3762,IT1:143_497(1.2),quarterly,Centro (I),index number,house price index (base 2015=100) - quarterly ...,H11 - new dwellings,2010-01-01,104.7,NaN
3642,IT1:143_497(1.2),quarterly,Centro (I),index number,house price index (base 2015=100) - quarterly ...,H1 - all items,2010-01-01,123.3,NaN
2067,IT1:143_497(1.2),quarterly,Torino,index number,house price index (base 2015=100) - quarterly ...,H1 - all items,2010-01-01,124.2,NaN
1542,IT1:143_497(1.2),quarterly,Nord-ovest,index number,house price index (base 2015=100) - quarterly ...,H1 - all items,2010-01-01,117.4,NaN
...,...,...,...,...,...,...,...,...,...
4286,IT1:143_497(1.2),quarterly,Roma,index number,house price index (base 2015=100) - quarterly ...,H12 - existing dwellings,2024-10-01,103.7,p
4226,IT1:143_497(1.2),quarterly,Roma,index number,house price index (base 2015=100) - quarterly ...,H1 - all items,2024-10-01,105.9,p
3821,IT1:143_497(1.2),quarterly,Centro (I),index number,house price index (base 2015=100) - quarterly ...,H11 - new dwellings,2024-10-01,127.0,p
3761,IT1:143_497(1.2),quarterly,Centro (I),index number,house price index (base 2015=100) - quarterly ...,H12 - existing dwellings,2024-10-01,103.1,p


In [41]:
import plotly.express as px
for abit_comprav in [
    "H12 - existing dwellings",
    "H11 - new dwellings"
]:
    fig = px.line(
        df3.query(f"ABIT_COMPRAV == '{abit_comprav}'").sort_values(by=["TIME_PERIOD", "ITTER107"]),
        x="TIME_PERIOD",
        y="OBS_VALUE",
        title=f"IPAB - {abit_comprav}",
        color="ITTER107"
    ).update_layout(width=1000).show()